## Transform Customer Data
1. Remove records with NULL customer_id
2. Remove extact duplicate records
3. Rmeove duplicate records based on created_timestampe
4. CAST the columns to the correct Data Type
5. Write transformed data to the Silver schema

### 1. Remove records with NULL customer_id


In [0]:
%sql
SELECT *
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

### 2. Remove extact duplicate records


In [0]:
%sql
SELECT DISTINCT *
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

### 3. Rmeove duplicate records based on created_timestampe


In [0]:
%sql
WITH cte_rank AS
(
  SELECT DISTINCT
    *,
    dense_rank() OVER (PARTITION BY customer_id ORDER BY created_timestamp DESC) AS rank
  FROM gizmobox.bronze.v_customers
  WHERE customer_id IS NOT NULL
)
SELECT *
FROM cte_rank
WHERE rank = 1
ORDER BY customer_id;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW tv_customer_distinct AS
SELECT DISTINCT *
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

In [0]:
%sql
WITH cte_max AS
(
  SELECT customer_id,
  MAX(created_timestamp) AS max_created_timestamp
  FROM tv_customer_distinct
  GROUP BY customer_id
)
SELECT t.*
FROM tv_customer_distinct t
JOIN cte_max m
ON t.customer_id = m.customer_id
AND t.created_timestamp = m.max_created_timestamp;

### 4. CAST the columns to the correct Data Type


In [0]:
%sql
WITH cte_max AS
(
  SELECT customer_id,
  MAX(created_timestamp) AS max_created_timestamp
  FROM tv_customer_distinct
  GROUP BY customer_id
)
SELECT 
  CAST(t.created_timestamp AS TIMESTAMP),
  t.customer_id,
  t.customer_name,
  CAST(t.date_of_birth AS DATE),
  t.email,
  CAST(t.member_since AS DATE),
  t.telephone,
  t.file_path
FROM tv_customer_distinct t
JOIN cte_max m
ON t.customer_id = m.customer_id
AND t.created_timestamp = m.max_created_timestamp;

### 5. Write transformed data to the Silver schema

In [0]:
CREATE TABLE gizmobox.silver.customers
AS
WITH cte_max AS
(
  SELECT customer_id,
  MAX(created_timestamp) AS max_created_timestamp
  FROM tv_customer_distinct
  GROUP BY customer_id
)
SELECT 
  CAST(t.created_timestamp AS TIMESTAMP),
  t.customer_id,
  t.customer_name,
  CAST(t.date_of_birth AS DATE),
  t.email,
  CAST(t.member_since AS DATE),
  t.telephone,
  t.file_path
FROM tv_customer_distinct t
JOIN cte_max m
ON t.customer_id = m.customer_id
AND t.created_timestamp = m.max_created_timestamp;

In [0]:
SELECT * FROM gizmobox.silver.customers;

In [0]:
DESCRIBE EXTENDED gizmobox.silver.customers;